# Week 66, Day 3: The GIL

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w066d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def finish_time(cpu_per_thread, threads, cores, gil):
    if gil:
        return cpu_per_thread * threads          # one thread runs bytecode at a time
    rounds = -(-threads // cores)                 # ceiling division: batches of `cores`
    return cpu_per_thread * rounds

for gil in (True, False):
    print("GIL" if gil else "no GIL", [finish_time(10, n, 4, gil) for n in (1, 2, 4, 8)])

In [ ]:
def round_robin(work, quantum):
    "Threads needing `work` units of CPU each, sharing one GIL in turns of `quantum`."
    remaining = dict(work)
    clock, finished = 0, {}
    while remaining:
        for name in list(remaining):
            step = min(quantum, remaining[name])
            clock += step
            remaining[name] -= step
            if remaining[name] == 0:
                finished[name] = clock
                del remaining[name]
    return finished

print(round_robin({"a": 10, "b": 10, "c": 10}, quantum=5))
print("alone, each would take 10; the last finishes at", max(round_robin({"a": 10, "b": 10, "c": 10}, 5).values()))

In [ ]:
counts = {"obj": 1}

def unsafe_incref_decref(schedule):
    "Two threads each do read-add-write on a reference count; schedule picks who runs each step."
    steps = {"A": ["read", "write+1"], "B": ["read", "write-1"]}
    local, pos = {}, {"A": 0, "B": 0}
    for who in schedule:
        action = steps[who][pos[who]]
        pos[who] += 1
        if action == "read":
            local[who] = counts["obj"]
        else:
            counts["obj"] = local[who] + (1 if action == "write+1" else -1)
    return counts["obj"]

counts["obj"] = 1
print("in turn:", unsafe_incref_decref("AABB"))
counts["obj"] = 1
print("interleaved:", unsafe_incref_decref("ABAB"), "(should be 1: an update was lost)")

In [ ]:
import sys

print(sys.getswitchinterval(), "seconds =", sys.getswitchinterval() * 1000, "ms")
old = sys.getswitchinterval()
sys.setswitchinterval(0.001)
print(sys.getswitchinterval())
sys.setswitchinterval(old)

def overlap(tasks):
    "Each task: (cpu, io). CPU parts need the GIL one at a time; I/O waits overlap freely."
    gil_free_at, finish = 0, {}
    for name, (cpu, io) in tasks.items():
        start = gil_free_at
        gil_free_at = start + cpu                 # holds the GIL while computing
        finish[name] = gil_free_at + io           # then waits without it
    return finish

print(overlap({"a": (1, 50), "b": (1, 50), "c": (1, 50)}))

In [ ]:
def run_increments(n, switch_every):
    "Two threads each do n increments as LOAD, ADD, STORE; switch threads every few instructions."
    counter = 0
    threads = {}
    for name in ("A", "B"):
        threads[name] = {"left": n, "step": 0, "value": None}
    order, turn, done = ["A", "B"], 0, 0
    while any(t["left"] for t in threads.values()):
        t = threads[order[turn % 2]]
        for _ in range(switch_every):
            if not t["left"]:
                break
            if t["step"] == 0:
                t["value"] = counter                 # LOAD
            elif t["step"] == 1:
                t["value"] += 1                      # ADD
            else:
                counter = t["value"]                 # STORE
                t["left"] -= 1
            t["step"] = (t["step"] + 1) % 3
        turn += 1
    return counter

print(run_increments(1000, switch_every=3000))   # switches only between whole runs: correct
print(run_increments(1000, switch_every=4))      # switches mid-increment: updates lost

In [ ]:
import sys

check = getattr(sys, "_is_gil_enabled", None)
if check is None:
    print("this Python is older than 3.13: it always has a GIL")
else:
    print("GIL enabled:", check())

def best_model(cpu_bound, free_threaded):
    if not cpu_bound:
        return "threads (or asyncio)"
    return "threads" if free_threaded else "processes"

print(best_model(True, False), "|", best_model(True, True), "|", best_model(False, False))

> **Key idea.** CPython's GIL lets only one thread run Python bytecode at a time per interpreter, so threads don't speed up CPU-bound Python code but do speed up I/O-bound code. It exists mainly to keep reference counting and the interpreter's data safe cheaply, which also made C extensions simple. A thread releases the GIL while blocked on I/O or waiting, inside long-running C code, and when asked after the switch interval (sys.getswitchinterval(), 0.005 seconds by default). The GIL doesn't make your logic thread-safe: += and check-then-act can still race, so use locks or queues. For parallel Python code use processes, the free-threaded build (3.13+, supported from 3.14; sys._is_gil_enabled()), or multiple interpreters (3.14); NumPy and other GIL-releasing C code already run in parallel.

### Exercise: Estimating with and without the GIL

Each task is `(name, cpu, io)`: it computes for `cpu` units, then waits `io` units. Write `with_gil(tasks)`: tasks are started in order, their CPU parts run **one at a time** (each task's CPU part starts when the previous one's ends), and each waits after its own CPU part; return a dict of finish times. Then `without_gil(tasks, cores)`: the same, but up to `cores` CPU parts run at once (give each task's CPU part to the core that's free first, the lowest-numbered on ties). With one core the two must agree.

In [ ]:
def with_gil(tasks):
    ___

def without_gil(tasks, cores):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_estimate():\n    \"GIL scheduling\"\n    io_tasks = [(\"a\", 1, 50), (\"b\", 1, 50), (\"c\", 1, 50)]\n    assert with_gil(io_tasks) == {\"a\": 51, \"b\": 52, \"c\": 53}\n    cpu_tasks = [(\"x\", 10, 0), (\"y\", 10, 0), (\"z\", 10, 0), (\"w\", 10, 0)]\n    assert with_gil(cpu_tasks) == {\"x\": 10, \"y\": 20, \"z\": 30, \"w\": 40}\n    assert without_gil(cpu_tasks, 4) == {\"x\": 10, \"y\": 10, \"z\": 10, \"w\": 10}\n    assert without_gil(cpu_tasks, 2) == {\"x\": 10, \"y\": 10, \"z\": 20, \"w\": 20}\n    assert without_gil(io_tasks, 1) == with_gil(io_tasks) and with_gil([]) == {}")

### Exercise: Taking turns on the GIL

Simulate CPU-bound threads sharing the GIL with a switch interval. `slices(work, quantum)`: `work` is a dict from thread name to the CPU time it needs; threads take turns in the dict's order, each running for up to `quantum` units before the next waiting thread gets the GIL. Return the list of `(name, start, end)` slices. Then `finish_times(work, quantum)` (a dict) and `switches(work, quantum)`: how many times the GIL passed from one thread to a **different** one.

In [ ]:
from collections import deque

def slices(work, quantum):
    ___

def finish_times(work, quantum):
    ___

def switches(work, quantum):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_quantum():\n    \"the switch interval\"\n    assert slices({\"a\": 7, \"b\": 3}, 5) == [(\"a\", 0, 5), (\"b\", 5, 8), (\"a\", 8, 10)]\n    assert finish_times({\"a\": 7, \"b\": 3}, 5) == {\"b\": 8, \"a\": 10}\n    assert finish_times({\"a\": 10, \"b\": 10, \"c\": 10}, 5) == {\"a\": 20, \"b\": 25, \"c\": 30}\n    assert switches({\"a\": 10, \"b\": 10}, 1) == 19 and switches({\"a\": 10, \"b\": 10}, 10) == 1\n    assert switches({\"solo\": 50}, 5) == 0 and slices({}, 5) == []")

### Exercise: The real switch interval

Write `interval_ms()`: the current switch interval in milliseconds, rounded to 3 places; a context manager `switch_interval(seconds)` that sets the interval for its block and restores the old value afterwards, even after an error; and `gil_state()`: `"on"` or `"off"` from `sys._is_gil_enabled()` when that exists, or `"on"` for older Pythons (which always have a GIL).

In [ ]:
import sys
from contextlib import contextmanager

def interval_ms():
    ___

@contextmanager
def switch_interval(seconds):
    ___

def gil_state():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_interval():\n    \"sys.getswitchinterval\"\n    import sys\n    before = sys.getswitchinterval()\n    assert interval_ms() == round(before * 1000, 3)\n    with switch_interval(0.002):\n        assert interval_ms() == 2.0\n    assert sys.getswitchinterval() == before\n    try:\n        with switch_interval(0.01):\n            raise KeyError\n    except KeyError:\n        pass\n    assert sys.getswitchinterval() == before and gil_state() in (\"on\", \"off\")")

### Exercise: The GIL doesn't make += atomic

Two threads each increment a shared counter `n` times; an increment is three instructions: `LOAD` (copy the counter into the thread's own register), `ADD` (add 1 to the register) and `STORE` (write the register back). Write `simulate(n, schedule)`: `schedule` is a string of `"A"` and `"B"` saying which thread runs **each instruction** (a thread with nothing left to do is skipped); return the final counter. Then `run_with_switch(n, k)`: build the schedule by switching between A and B every `k` instructions, starting with A, and simulate it, and `lost(n, k)`: how many increments were lost.

In [ ]:
def simulate(n, schedule):
    ___

def run_with_switch(n, k):
    ___

def lost(n, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lost():\n    \"race conditions under the GIL\"\n    assert simulate(1, \"AAABBB\") == 2 and simulate(1, \"ABABAB\") == 1\n    assert simulate(2, \"AAAAAABBBBBB\") == 4 and simulate(1, \"AABBBA\") == 1\n    assert run_with_switch(100, 300) == 200 and lost(100, 300) == 0\n    assert lost(100, 3) == 0 and lost(100, 2) > 0 and lost(100, 4) > 0\n    assert simulate(1, \"A\" * 10 + \"B\" * 3) == 2")

### Exercise: Choosing a way around the GIL

Write `recommend(work)` for a dict describing a workload, returning one of `"single thread"`, `"threads"`, `"processes"`, or `"numpy"`: if `work["tasks"]` is 1, `"single thread"`; otherwise, if the work is I/O-bound (`work["io_bound"]`), `"threads"`; otherwise, if it's numeric array work (`work["numeric"]`), `"numpy"`; otherwise `"threads"` if the interpreter is free-threaded (`work["free_threaded"]`) and `"processes"` if not. Then `explain(choice)`: a short reason from this table: single thread, "nothing to overlap"; threads, "waiting overlaps, or no GIL"; processes, "one GIL per process"; numpy, "work runs in C outside the GIL".

In [ ]:
def recommend(work):
    ___

def explain(choice):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_choose():\n    \"choosing a model\"\n    base = {\"tasks\": 8, \"io_bound\": False, \"numeric\": False, \"free_threaded\": False}\n    assert recommend(dict(base, tasks=1)) == \"single thread\"\n    assert recommend(dict(base, io_bound=True)) == \"threads\"\n    assert recommend(dict(base, numeric=True)) == \"numpy\"\n    assert recommend(base) == \"processes\" and recommend(dict(base, free_threaded=True)) == \"threads\"\n    assert explain(\"processes\") == \"one GIL per process\" and explain(\"numpy\") == \"work runs in C outside the GIL\"")

### Bug hunt: CPU work that overlapped by magic

The estimate treats CPU work like waiting, letting it overlap, so four CPU-bound threads appear to finish in the time of one: this prints `10`. With the GIL, their CPU parts run one after another, so the total should be the **sum**: make it print `40`.

In [ ]:
cpu_parts = [10, 10, 10, 10]
print(max(cpu_parts))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"CPU parts serialise\"\n    expect_output(\"40\")")

### Bug hunt: Milliseconds or seconds?

`sys.setswitchinterval` takes **seconds**, so passing `5` sets the interval to five whole seconds and this prints `5000.0`. Pass the value in seconds so it prints `5.0`, and restore the old interval afterwards.

In [ ]:
import sys

old = sys.getswitchinterval()
sys.setswitchinterval(5)
print(sys.getswitchinterval() * 1000)
sys.setswitchinterval(old)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"seconds\"\n    expect_output(\"5.0\")")

### Bug hunt: Turns that never end

In this round-robin, a thread that still has work left is put back at the **front** of the queue, so it keeps the GIL until it's finished and the others wait: this prints `{'a': 10, 'b': 15}`. Put unfinished threads at the **back** so they take turns, and it prints `{'b': 10, 'a': 15}`.

In [ ]:
from collections import deque

queue = deque([("a", 10), ("b", 5)])
clock, done = 0, {}
while queue:
    name, need = queue.popleft()
    step = min(5, need)
    clock += step
    if need > step:
        queue.appendleft((name, need - step))
    else:
        done[name] = clock
print(done)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"back of the queue\"\n    expect_output(\"{'b': 10, 'a': 15}\")")

### Puzzle: Restoring the switch interval

Arrange the lines of a context manager that changes the switch interval for a block and always restores it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: with the GIL, two CPU-bound Python threads run at the same instant
q = ___   # the default switch interval in milliseconds, as a number
r = ___   # True or False: a thread waiting on a socket holds the GIL
s = ___   # True or False: the GIL makes counter += 1 safe across threads
t = ___   # the Python version that first offered a free-threaded build, as a string like "3.10"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == 5 and r is False, \"p, q and r\"\n    assert s is False and t == \"3.13\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a GIL simulator

Simulate threads under a GIL with a switch interval, and compare with a free-threaded interpreter. Each thread is `(name, steps)`, with steps `("cpu", t)` and `("io", t)`. Write `simulate(threads, quantum, cores=1, gil=True)` returning a dict of finish times:

- all threads are ready at time 0, in the order given;
- **with the GIL**, only one thread runs CPU work at a time, whatever `cores` is: it runs for at most `quantum` units, then, if any other thread is ready, goes to the back of the ready queue (otherwise it simply carries on);
- **without the GIL**, up to `cores` threads run CPU work at once, each for up to `quantum` units before going to the back of the queue if others are waiting;
- an I/O step needs no GIL and no core: the thread waits for it and then rejoins the back of the ready queue (or finishes);
- events at the same time are handled in this order: threads finishing I/O rejoin the queue first, then threads whose time slice ended.

Advance time in steps of one unit (simplest to get right), or from event to event.

In [ ]:
from collections import deque

def simulate(threads, quantum, cores=1, gil=True):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gil():\n    \"with the GIL\"\n    cpu = [(\"a\", [(\"cpu\", 4)]), (\"b\", [(\"cpu\", 4)])]\n    assert simulate(cpu, quantum=2) == {\"a\": 6, \"b\": 8}\n    assert simulate(cpu, quantum=2, cores=4) == {\"a\": 6, \"b\": 8}\n    assert simulate(cpu, quantum=10) == {\"a\": 4, \"b\": 8}\n    io = [(\"a\", [(\"cpu\", 1), (\"io\", 10)]), (\"b\", [(\"cpu\", 1), (\"io\", 10)])]\n    assert simulate(io, quantum=5) == {\"a\": 11, \"b\": 12}\n    mixed = [(\"calc\", [(\"cpu\", 6)]), (\"net\", [(\"io\", 3), (\"cpu\", 1)])]\n    assert simulate(mixed, quantum=2) == {\"net\": 5, \"calc\": 7}\n\ndef test_free_threaded():\n    \"without the GIL\"\n    cpu = [(\"a\", [(\"cpu\", 4)]), (\"b\", [(\"cpu\", 4)]), (\"c\", [(\"cpu\", 4)])]\n    assert simulate(cpu, quantum=2, cores=3, gil=False) == {\"a\": 4, \"b\": 4, \"c\": 4}\n    assert simulate(cpu, quantum=4, cores=2, gil=False) == {\"a\": 4, \"b\": 4, \"c\": 8}\n    assert simulate(cpu, quantum=2, cores=1, gil=False) == simulate(cpu, quantum=2)\n    assert simulate([], 5) == {}")

## Warm-up

With the GIL, four CPU-bound threads take four times as long as one: the work is shared out, but never runs at the same time. Without it, four threads on four cores finish together. (Real numbers are a little worse with the GIL, because switching between threads has a cost.)

## One thread at a time

The **global interpreter lock** is a lock inside the CPython interpreter. A thread must hold it to execute Python bytecode or touch Python objects. So, within one process:

- **Only one thread runs Python code at any instant**, even on a machine with many cores. Threads take turns.
- **CPU-bound Python code gets no faster with threads**, and often gets a little slower, because of the cost of switching.
- **I/O-bound code still benefits**, because a thread that's waiting doesn't need the GIL (next sections): while one thread waits for a network reply, another runs.

The GIL belongs to CPython (the standard interpreter), not to the Python language. PyPy also has one; Jython and IronPython never did. And since Python 3.13 there's an optional build of CPython without it.

## Why CPython has a GIL

The main reason is **reference counting** (Week 62). Every time any code creates or drops a reference, it changes an object's `ob_refcnt`. If two threads did that to the same object at the same time without coordination, counts could be lost, and objects would be freed while still in use (a crash) or never freed (a leak). Protecting every count with its own lock would make **all** code slower, including the vast majority of programs that use one thread.

One big lock is:

- **fast for single-threaded programs**: acquiring it rarely is cheap;
- **simple for C extensions**: code holding the GIL can use Python objects without worrying about other threads, which made it easy for the ecosystem of C extensions (NumPy and many others) to grow;
- **easy to get right** inside the interpreter.

The cost is the one you saw: no parallel execution of Python code in threads. For a long time this was considered the right trade-off; removing the GIL without slowing everything down took many years of work (next sections).

## When the GIL is released

A thread gives up the GIL:

- **when it blocks on I/O or waits**: reading a file or socket, `time.sleep`, waiting for a lock, a queue or `join`. C code wraps the blocking call in `Py_BEGIN_ALLOW_THREADS` / `Py_END_ALLOW_THREADS`, releasing the GIL for its duration. That's why threads speed up I/O-bound programs;
- **in long-running C code that doesn't touch Python objects**: `hashlib` on large data, `zlib` compression, many NumPy operations. Such code can run in parallel with Python code in other threads;
- **when asked to**: a thread that's been running Python code for longer than the **switch interval** is asked to release the GIL so a waiting thread can have a turn. The interval is 5 milliseconds by default: `sys.getswitchinterval()` returns it in **seconds** (`0.005`) and `sys.setswitchinterval(s)` changes it. A shorter interval makes threads more responsive but adds switching overhead.

The interpreter checks for that request regularly in the evaluation loop (Week 64), between bytecode instructions, which is why a thread can be paused between almost **any** two instructions.

## What the GIL doesn't do

A common misunderstanding is that the GIL makes threaded code **safe**. It doesn't. It protects the **interpreter's** internal data (reference counts, the insides of lists and dicts) so that CPython itself doesn't crash, but your program's **logic** can still go wrong:

- A thread can be paused between any two bytecode instructions. `counter += 1` is several instructions (load, add, store), so two threads can both load the same old value and one increment is **lost**: a **race condition** (tomorrow).
- **Check-then-act** sequences (`if key not in cache: cache[key] = compute()`) can interleave too.
- Some single operations happen to be atomic in CPython (`list.append`, `dict[key] = value`, `x = y`), but that's an implementation detail, and it no longer holds in the free-threaded build in the same way. Use **locks** (tomorrow) or **queues** for any data shared between threads.

## Life without the GIL

When you need Python code to run on several cores at once, there are three routes:

- **Processes** (`multiprocessing`, `ProcessPoolExecutor`): each process has its own interpreter and its own GIL, so they run truly in parallel. The costs are start-up time, more memory, and copying data between processes (Monday). This is the standard answer for CPU-bound Python work.
- **Free-threaded Python** (PEP 703): Python 3.13 introduced an optional build of CPython without the GIL (`python3.13t`), and Python 3.14 made it officially supported (PEP 779). It uses different reference-counting techniques (biased and deferred counting, and immortal objects) and finer-grained locks inside lists and dicts. Single-threaded code runs somewhat slower than in the normal build, and C extensions must be updated to declare they're safe without the GIL. `sys._is_gil_enabled()` (3.13+) tells you whether the GIL is on.
- **Multiple interpreters** (PEP 734, Python 3.14's `concurrent.interpreters`): several isolated interpreters in one process, each with its own GIL, communicating through queues. Lighter than processes, still isolated.

Also: work done inside NumPy or other C code that releases the GIL already runs in parallel in threads.

On your own computer, compare threads and processes on CPU-bound work:

```
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def busy(n):
    return sum(i * i for i in range(n))

if __name__ == "__main__":
    for Pool in (ThreadPoolExecutor, ProcessPoolExecutor):
        start = time.perf_counter()
        with Pool(max_workers=4) as pool:
            list(pool.map(busy, [2_000_000] * 4))
        print(Pool.__name__, round(time.perf_counter() - start, 2), "s")
```

## Practice

## Quiz

## Daily challenge

## Before you go

If you have Python 3.13 or later on your own computer, check whether a free-threaded build is available for it (on python.org's installers it's an optional extra), and run the thread-versus-process comparison with both. Tomorrow: race conditions, and the locks that prevent them.